# 🔍 Notebook 01: Data Profiling & Quality Exploration
**โครงการ**: Used Car Analytics (Data Warehouse & ETL Pipeline)

วัตถุประสงค์ของ Notebook นี้:
1. สำรวจข้อมูลดิบจาก 3 Data Sources หลักรวมถึงไฟล์สกัดหลายช่วงเวลา (Multi-file Web Scraped Series)
2. แปลงและปรับมาตรฐานชื่อคอลัมน์ดิบจาก Web Scraper (`data`, `data2`, `data3`... -> `car_title`, `description`...)
3. ตรวจสอบปัญหาคุณภาพข้อมูล (Data Quality Issues) 5 ประเด็นสำคัญตามเกณฑ์อาจารย์

In [1]:
import pandas as pd
import glob
import os

# 1. Ingest Multi-file One2car Scraped Series
raw_one2car_files = sorted(glob.glob('../../01_Raw_Data/one2car/one2car-11-*.csv'))
print(f'Found {len(raw_one2car_files)} scraped period files: {raw_one2car_files}')

def standardize_scraped_columns(df):
    rename_map = {
        'data': 'car_title',
        'data2': 'description',
        'data3': 'mileage',
        'data4': 'location',
        'data6': 'car_model',
        'data16': 'transmission'
    }
    return df.rename(columns=rename_map)

df_list = [standardize_scraped_columns(pd.read_csv(f)) for f in raw_one2car_files]
df_one2car = pd.concat(df_list, ignore_index=True)

print('One2car Consolidated Data Shape:', df_one2car.shape)
df_one2car.head(3)

Found 3 scraped period files: ['../../01_Raw_Data/one2car/one2car-11-2.csv', '../../01_Raw_Data/one2car/one2car-11-3.csv', '../../01_Raw_Data/one2car/one2car-11-4.csv']
One2car Consolidated Data Shape: (4190, 15)


,web_scraper_order,web_scraper_start_url,pagination,car_title,price,description,mileage,location,rating,car_model,transmission,data12,data17,data15,data22
0,1786479238-1,https://www.one2car.com/%E0%B8%A3%E0%B8%96%E0%...,NaN,2015 Honda City 1.5 (ปี 14-18) SV+ Sedan - SV,"269,000 บาท","HONDA CITY, 1.5 SV+ i-VTEC 2015 รถบ้านแท้มือเด...",170 - 175K กม.,กรุงเทพมหานคร,Honda City,มือสอง,เกียร์อัตโนมัติ,NaN,NaN,NaN,NaN
1,1786479238-2,https://www.one2car.com/%E0%B8%A3%E0%B8%96%E0%...,NaN,2023 Honda City 1.0 (ปี 19-26) SV Sedan,"359,000 บาท",'23 Honda City SV 1.0 Turbo ตัว Top รถมือเดียว...,30 - 35K กม.,สมุทรปราการ,Honda City,มือสอง,เกียร์อัตโนมัติ,NaN,NaN,NaN,NaN
2,1786479238-3,https://www.one2car.com/%E0%B8%A3%E0%B8%96%E0%...,NaN,2025 BMW 220i 2.0 F44 (ปี 20-27) Gran M Sport ...,"1,250,000 บาท",💢เจ้าของขายเองครับ💢 '25 แท้ BMW 220i M sport T...,30 - 35K กม.,สมุทรปราการ,NaN,มือสอง,เกียร์อัตโนมัติ,NaN,NaN,NaN,NaN


In [2]:
# 2. Ingest US Sales & Spec Datasets
df_us_sales = pd.read_csv('../../01_Raw_Data/us-usecar/used_car_sales.csv')
df_spec1 = pd.read_csv('../../01_Raw_Data/usecar-dataset/used_car_dataset.csv')
df_spec2 = pd.read_csv('../../01_Raw_Data/usecar-dataset/used_cars_dataset_2.csv')
df_spec = pd.concat([df_spec1, df_spec2], ignore_index=True)

print('US Sales Shape:', df_us_sales.shape)
print('Spec Combined Shape:', df_spec.shape)

US Sales Shape: (122144, 13)
Spec Combined Shape: (24575, 11)


In [3]:
# 3. Audit Data Quality Issues (5 Issues Identified)
print('--- DQ Issue 1: Missing Prices in One2car ---')
print(f'Null prices: {df_one2car["price"].isnull().sum()} / {len(df_one2car)}')

print('\n--- DQ Issue 2: Price String Formatting & Symbols ---')
print(df_one2car['price'].dropna().head(5))

print('\n--- DQ Issue 3: Mileage Range Formats (e.g. 170 - 175K กม.) ---')
print(df_one2car['mileage'].dropna().head(5))

print('\n--- DQ Issue 4: Unstructured Text Title Parsing ---')
print(df_one2car['car_title'].dropna().head(5))

--- DQ Issue 1: Missing Prices in One2car ---
Null prices: 223 / 4190

--- DQ Issue 2: Price String Formatting & Symbols ---
0      269,000 บาท
1      359,000 บาท
2    1,250,000 บาท
3      449,000 บาท
4      259,000 บาท
Name: price, dtype: str

--- DQ Issue 3: Mileage Range Formats (e.g. 170 - 175K กม.) ---
0    170 - 175K กม.
1      30 - 35K กม.
2      30 - 35K กม.
3    110 - 115K กม.
4      20 - 25K กม.
Name: mileage, dtype: str

--- DQ Issue 4: Unstructured Text Title Parsing ---
0        2015 Honda City 1.5 (ปี 14-18) SV+ Sedan - SV
1              2023 Honda City 1.0 (ปี 19-26) SV Sedan
2    2025 BMW 220i 2.0 F44 (ปี 20-27) Gran M Sport ...
3    2022 Toyota HILUX REVO 2.4 Double Cab Z Editio...
4              2015 Honda City 1.5 (ปี 14-18) SV Sedan
Name: car_title, dtype: str
